# Phase 4 GCC Constraints — Analysis

Compares Phase 3 (no GCC) vs Phase 4 (GCC-constrained) fault injection results.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("../../")))

from analysis.aggregate import load_runs
from analysis.plots import (
    plot_recovery_modes, plot_fault_latency,
    plot_phase3_vs_phase4,
)
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

p3 = load_runs("../../artifacts/", prefix="failures")
p4 = load_runs("../../artifacts/", prefix="gcc")
print(f"Phase 3: {len(p3)} runs, Phase 4: {len(p4)} runs")

In [ ]:
# Phase 3 recovery modes
fig = plot_recovery_modes(p3, label="Phase 3 (no GCC)")
plt.show()

In [ ]:
# Phase 4 recovery modes (GCC-constrained)
fig = plot_recovery_modes(p4, label="Phase 4 (GCC)")
plt.show()

In [ ]:
# Phase 3 vs Phase 4 latency comparison
fig = plot_phase3_vs_phase4(p3, p4)
plt.show()

In [ ]:
# Phase 4 latency distribution by fault type
fig = plot_fault_latency(p4, label="Phase 4 (GCC)")
plt.show()

In [ ]:
# Detailed comparison table
comparison_rows = []
for ft in sorted(p3["failure_type"].dropna().unique()):
    p3_sub = p3[p3["failure_type"] == ft]
    p4_sub = p4[p4["failure_type"] == ft]
    comparison_rows.append({
        "fault_type": ft,
        "p3_runs": len(p3_sub),
        "p3_avg_latency": p3_sub["total_latency_s"].mean(),
        "p3_recovery": p3_sub["recovery_mode"].mode().iloc[0] if len(p3_sub) > 0 else "n/a",
        "p4_runs": len(p4_sub),
        "p4_avg_latency": p4_sub["total_latency_s"].mean(),
        "p4_recovery": p4_sub["recovery_mode"].mode().iloc[0] if len(p4_sub) > 0 else "n/a",
        "latency_delta": p4_sub["total_latency_s"].mean() - p3_sub["total_latency_s"].mean(),
        "degraded": p3_sub["recovery_mode"].mode().iloc[0] != p4_sub["recovery_mode"].mode().iloc[0] if len(p3_sub) > 0 and len(p4_sub) > 0 else False,
    })
comp_df = pd.DataFrame(comparison_rows).round(3)
display(comp_df)

## Observations

### Recovery Mode Stability
All four fault types maintain the same recovery mode under GCC constraints:
- `malformed_json` stays automatic (crew_node catches it regardless of latency)
- `timeout`, `checkpoint_loss`, `context_overflow` stay manual

### GCC Latency Impact
- **malformed_json** shows clearest overhead: ~0.002s → ~0.23s (200ms GCC latency visible)
- **checkpoint_loss** has highest absolute overhead: +3-5s on top of ~68s base (crew runs fully)
- **timeout** adds ~0.2s (GCC latency fires before the pre-sleep timeout)
- **context_overflow** adds ~0.2s (GCC latency before token check)

### TPM Rate Cap
The 60k TPM cap was never triggered — workspace data is too small (~410 tokens per run).
In production with real document volumes, this would be the binding constraint.

### Key Conclusion
GCC constraints add latency overhead but don't change failure recovery modes for
these four fault types. The real GCC risk is on *non-injected* failures: genuine
timeouts from slow endpoints under 200ms+ added latency would be more frequent,
and TPM caps would throttle high-volume workloads.